# Child Sleep & Development
## Apache Spark: Lazy Evaluation & Partitioning — Live Demo

Based on the **National Survey of Children's Health** (CDC, 2016–2022).  
Real schema and statistical distributions, scaled to 5 million records.

> *Your 3-year-old won't sleep through the night. You found this dataset.*  
> *700,000 kids surveyed. Scaled to 5 million. You open Spark and start asking questions.*

---

| Demo | Spark concepts |
|------|----------------|
| **Demo 1 — Lazy Evaluation & Catalyst** | Transformations vs actions, Catalyst optimizer, `explain()` as validation |
| **Demo 2 — Partitioning** | On-disk partitioning, partition pruning, repartition vs coalesce, data skew & salting |
| **Demo 3 — Caching & UDFs** | Caching, `explain()` as diagnostic tool, UDFs vs built-ins |

---

| Section | Story beat | Spark concept |
|---------|-----------|---------------|
| Demo 1 — Part 1 | The first look | Transformations vs actions |
| Demo 1 — Part 2 | The Monday report | Cost of multiple actions |
| Demo 1 — Part 3 | The wellness score | Catalyst optimizer proof |
| Demo 1 — Part 4 | What is Spark planning? | `explain()` as validation |
| Demo 2 — Part 5 | Why are we scanning all 50 states? | Partition pruning |
| Demo 2 — Part 6 | Reshaping partitions | repartition vs coalesce |
| Demo 2 — Part 7 | California is breaking your groupBy | Data skew + salting |
| Demo 3 — Part 8 | Three questions, one dataset | Caching |
| Demo 3 — Part 9 | The sleep classifier | UDFs vs built-ins + `explain()` |


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as spark_sum, avg, count, when, lit,
    rand, expr, round as spark_round
)
from pathlib import Path
import time, os

spark = SparkSession.builder \
    .appName('ChildHealthDemo') \
    .config('spark.driver.memory', '4g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')

# Docker: /app/data  |  Local: demo_data/ next to the notebook
if os.path.isdir('/app/data'):
    BASE = Path('/app/data')
else:
    BASE = Path.cwd() / 'demo_data'
    os.makedirs(str(BASE), exist_ok=True)

DATA = (BASE / 'nsch_extended').as_posix()
PART = (BASE / 'nsch_by_state').as_posix()

print('Spark', spark.version, '| UI: http://localhost:4040')
print('Data :', DATA)
# ── pretty output helper ─────────────────────────────────────────────────────
from IPython.display import display, HTML

def show(title, rows, accent='#2563eb', bg='#f4f6fb'):
        """Render a styled result box in Jupyter output."""
        tds = ''.join(
            "<tr><td style='padding:3px 16px;color:#1e293b;"
            "font-family:monospace;font-size:14px;white-space:pre;text-align:left'>{}</td></tr>".format(r)
            for r in rows
        )
        html = (
            "<div style='background:{};border-left:4px solid {};"
            "border-radius:4px;padding:10px 0 6px;margin:8px 0'>"
            "<div style='color:{};font-family:monospace;font-size:12px;"
            "font-weight:bold;padding:0 16px 8px'>{}</div>"
            "<table style='border-collapse:collapse;width:100%'>{}</table></div>"
        ).format(bg, accent, accent, title, tds)
        display(HTML(html))


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/07 20:27:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.1 | UI: http://localhost:4040
Data : /app/data/nsch_extended


In [ ]:
---
## Setup: Generating the Dataset

Run once — takes ~1 min. Skipped automatically on reruns.

**Schema** (mirrors NSCH variables for children 0–5 years):
```
child_id, survey_year, age_months, sex, state, census_region,
health_status, sleep_hours, adequate_sleep, has_regular_bedtime,
screen_time_hours, outdoor_play_hours, reads_together,
flourishing_affection, flourishing_resilience, flourishing_curiosity,
daycare_attendance, developmental_concern
```
Distributions match published NSCH statistics:
85% have a regular bedtime, 60% meet sleep guidelines, flourishing scores skew positive.

In [ ]:
FORCE_REGEN = False  # set True to regenerate

if FORCE_REGEN or not os.path.exists(DATA):
    print('Generating 5M records based on NSCH distributions...')
    t0 = time.time()

    # States ordered by census region (Northeast first, then Midwest, South, West)
    # This lets us derive region from state_idx with simple threshold comparisons
    northeast = ['CT','ME','MA','NH','RI','VT','NJ','NY','PA']           # idx 0-8
    midwest   = ['IL','IN','MI','OH','WI','IA','KS','MN','MO','NE','ND','SD']  # idx 9-20
    south     = ['DE','FL','GA','MD','NC','SC','VA','WV','AL',
                 'KY','MS','TN','AR','LA','OK','TX']                     # idx 21-36
    west      = ['AZ','CO','ID','MT','NV','NM','UT','WY',
                 'AK','CA','HI','OR','WA']                               # idx 37-49
    all_states = northeast + midwest + south + west

    def case_map(idx_col, values):
        parts = ['WHEN {} = {} THEN {!r}'.format(idx_col, i, v) for i, v in enumerate(values)]
        return 'CASE {} ELSE {!r} END'.format(' '.join(parts), values[0])

    raw = spark.range(5_000_000) \
        .withColumn('r_state',   rand()) \
        .withColumn('r_sex',     rand()) \
        .withColumn('r_health',  rand()) \
        .withColumn('r_bedtime', rand()) \
        .withColumn('r_sleep',   rand()) \
        .withColumn('r_screen',  rand()) \
        .withColumn('r_outdoor', rand()) \
        .withColumn('r_reads',   rand()) \
        .withColumn('r_aff',     rand()) \
        .withColumn('r_res',     rand()) \
        .withColumn('r_cur',     rand()) \
        .withColumn('r_daycare', rand()) \
        .withColumn('r_devcon',  rand())

    raw = raw \
        .withColumn('state_idx', (col('r_state') * 50).cast('int')) \
        .withColumn('state',         expr(case_map('state_idx', all_states))) \
        .withColumn('census_region',
            when(col('state_idx') < 9,  'Northeast')
            .when(col('state_idx') < 21, 'Midwest')
            .when(col('state_idx') < 37, 'South')
            .otherwise('West')
        ) \
        .withColumn('age_months',   (col('r_state') * 60).cast('int')) \
        .withColumn('survey_year',  (col('r_screen') * 7).cast('int') + 2016) \
        .withColumn('sex',          when(col('r_sex') > 0.5, 'Male').otherwise('Female')) \
        .withColumn('health_status',
            when(col('r_health') < 0.40, 'Excellent')
            .when(col('r_health') < 0.75, 'Very_Good')
            .when(col('r_health') < 0.93, 'Good')
            .when(col('r_health') < 0.98, 'Fair')
            .otherwise('Poor')
        ) \
        .withColumn('has_regular_bedtime', when(col('r_bedtime') > 0.15, 'Yes').otherwise('No')) \
        .withColumn('sleep_hours',
            spark_round(
                when(col('has_regular_bedtime') == 'Yes', 9.0 + col('r_sleep') * 4.5)
                .otherwise(6.0 + col('r_sleep') * 3.5)
            , 1)
        ) \
        .withColumn('adequate_sleep', when(col('sleep_hours') >= 10, 'Yes').otherwise('No')) \
        .withColumn('screen_time_hours',  spark_round(col('r_screen') * 4.5, 1)) \
        .withColumn('outdoor_play_hours', spark_round(0.5 + col('r_outdoor') * 3.5, 1)) \
        .withColumn('reads_together',
            when(col('r_reads') < 0.50, 'Daily')
            .when(col('r_reads') < 0.75, 'Several_times_week')
            .when(col('r_reads') < 0.90, 'Once_a_week')
            .otherwise('Rarely')
        ) \
        .withColumn('flourishing_affection',
            when(col('r_aff') < 0.70, 'Always')
            .when(col('r_aff') < 0.90, 'Usually')
            .when(col('r_aff') < 0.97, 'Sometimes')
            .otherwise('Rarely')
        ) \
        .withColumn('flourishing_resilience',
            when(col('r_res') < 0.55, 'Always')
            .when(col('r_res') < 0.85, 'Usually')
            .when(col('r_res') < 0.95, 'Sometimes')
            .otherwise('Rarely')
        ) \
        .withColumn('flourishing_curiosity',
            when(col('r_cur') < 0.65, 'Always')
            .when(col('r_cur') < 0.90, 'Usually')
            .when(col('r_cur') < 0.97, 'Sometimes')
            .otherwise('Rarely')
        ) \
        .withColumn('daycare_attendance',   when(col('r_daycare') > 0.45, 'Yes').otherwise('No')) \
        .withColumn('developmental_concern', when(col('r_devcon') > 0.85, 'Yes').otherwise('No')) \
        .select(
            col('id').alias('child_id'), 'survey_year', 'age_months', 'sex',
            'state', 'census_region', 'health_status',
            'sleep_hours', 'adequate_sleep', 'has_regular_bedtime',
            'screen_time_hours', 'outdoor_play_hours', 'reads_together',
            'flourishing_affection', 'flourishing_resilience', 'flourishing_curiosity',
            'daycare_attendance', 'developmental_concern'
        )

    raw.write.mode('overwrite').parquet(DATA)
    print('Done in {:.1f}s  ->  {}'.format(time.time() - t0, DATA))
else:
    print('Data already at', DATA)
    print('(set FORCE_REGEN = True to regenerate)')

In [2]:
df = spark.read.parquet(DATA)

# ── schema ───────────────────────────────────────────────────────────────
fields = df.schema.fields
rows = ''.join(
    "<tr style='border-bottom:1px solid #e2e8f0'>"
    "<td style='padding:4px 16px;color:#1e293b;font-family:monospace;font-size:13px;text-align:left'>{}</td>"
    "<td style='padding:4px 16px;color:#16a34a;font-family:monospace;font-size:13px;text-align:left'>{}</td>"
    "<td style='padding:4px 16px;color:#94a3b8;font-family:monospace;font-size:12px;text-align:left'>{}</td>"
    "</tr>".format(f.name, f.dataType.simpleString(), 'nullable' if f.nullable else 'required')
    for f in fields
)
n_parts = df.rdd.getNumPartitions()
display(HTML(
    "<div style='background:#f4f6fb;border-left:4px solid #2563eb;border-radius:4px;padding:10px 0 6px;margin:8px 0'>"
    "<div style='color:#2563eb;font-family:monospace;font-size:12px;font-weight:bold;padding:0 16px 8px'>SCHEMA &mdash; {} columns &mdash; {} partitions</div>"
    "<table style='border-collapse:collapse;width:auto'>{}</table></div>"
    .format(len(fields), n_parts, rows)
))

# ── sample rows (Jupyter renders pandas as a styled HTML table) ──────────
display(df.limit(5).toPandas())


child_id,bigint,nullable
survey_year,int,nullable
age_months,int,nullable
sex,string,nullable
state,string,nullable
census_region,string,nullable
health_status,string,nullable
sleep_hours,double,nullable
adequate_sleep,string,nullable
has_regular_bedtime,string,nullable
screen_time_hours,double,nullable


,child_id,survey_year,age_months,sex,state,census_region,health_status,sleep_hours,adequate_sleep,has_regular_bedtime,screen_time_hours,outdoor_play_hours,reads_together,flourishing_affection,flourishing_resilience,flourishing_curiosity,daycare_attendance,developmental_concern
0,4166666,2017,44,Male,AZ,West,Very_Good,10.7,Yes,Yes,1.3,2.6,Rarely,Sometimes,Always,Usually,No,No
1,4166667,2022,26,Female,FL,South,Excellent,9.2,No,Yes,4.2,1.7,Daily,Always,Rarely,Rarely,No,Yes
2,4166668,2021,12,Female,IN,Midwest,Excellent,11.4,Yes,Yes,3.7,0.7,Rarely,Usually,Always,Always,Yes,No
3,4166669,2022,12,Male,IN,Midwest,Good,9.1,No,No,4.4,3.2,Once_a_week,Always,Always,Always,No,No
4,4166670,2020,41,Male,LA,South,Very_Good,9.5,No,Yes,3.1,2.7,Once_a_week,Always,Usually,Always,No,No


---
# 🔵 DEMO 1 — Lazy Evaluation & Catalyst Optimizer

**What we’re demonstrating:**
- Transformations vs actions — Spark queues work and waits
- The cost of calling too many actions
- Column pruning — Catalyst eliminates unused work automatically
- Reading execution plans with `explain()`

**Key thing to watch:**
- The transformation chain takes milliseconds; `.count()` takes seconds
- Creating AND dropping columns is the same speed as doing nothing
- Four `.filter()` calls collapse into one in the optimized plan


---
# Part 1 — The First Look

You open the dataset. You want to understand how toddlers in the Northeast are sleeping.  
You write a chain of questions — one step at a time.

**Concept: Transformations vs Actions**
- `filter()`, `select()`, `groupBy()` = **transformations** — they build the plan, nothing runs
- `count()`, `collect()`, `show()` = **actions** — they trigger execution of the whole plan
- Spark queues all transformations and runs them together when an action is called

**What to notice:**
- The transformation chain completes in milliseconds — no data was touched
- `.count()` triggers everything at once: all 5 operations in a single optimized pass
- This is what "lazy evaluation" means: describe the work, Spark decides when and how to do it

In [3]:
t_start = time.time()

# filter() = WHERE in SQL. col() = column reference.
step1 = df.filter(col('census_region') == 'Northeast')
step2 = step1.filter(col('age_months') <= 36)
step3 = step2.filter(col('adequate_sleep') == 'No')
# select() = SELECT in SQL -- pick only the columns you need
step4 = step3.select('child_id', 'age_months', 'sleep_hours', 'state', 'has_regular_bedtime')
# groupBy().agg() = GROUP BY with aggregations, same as SQL
step5 = step4.groupBy('state').agg(
    avg('sleep_hours').alias('avg_sleep'),
    count('*').alias('children')
)

t_chain = time.time() - t_start
show('TRANSFORMATIONS — nothing ran yet', [
    '5 operations built in <b style="color:#16a34a">{:.4f}s</b>'.format(t_chain),
    'filter → filter → filter → select → groupBy',
    '',
    'Spark wrote down all five questions. The data was never touched.',
])


5 operations built in 0.3870s
filter → filter → filter → select → groupBy
""
Spark wrote down all five questions. The data was never touched.


In [4]:
# .count() is an action -- the trigger that executes everything
a_start = time.time()
result = step5.count()
t_action = time.time() - a_start

show('ACTION — .count() triggered everything', [
    'Northeast states in dataset:        <b style="color:#16a34a">{}</b>'.format(result),
    'Time to run all 5 transformations:  <b style="color:#dc2626">{:.2f}s</b>'.format(t_action),
    '',
    'That pause: Spark read 5M records and ran every transformation',
    'in a single optimised pass — not one step at a time.',
])


Northeast states in dataset: 9
Time to run all 5 transformations: 1.96s
""
That pause: Spark read 5M records and ran every transformation
in a single optimised pass — not one step at a time.


---
# Part 2 — The Monday Report

You want the full picture: average sleep, bedtime regularity, adequate sleep rate, regional breakdown.  
First attempt — the obvious way, one question per line.

**Concept: Cost of multiple actions**
- Every `.count()` and `.collect()` is an action — each one re-executes the full plan from scratch
- 5 separate actions = 5 full scans of 5 million rows
- Solution: combine multiple metrics into a single `.agg()` call — one action, one scan

**What to notice:**
- Version 1 has 5 actions: `count`, `agg+collect`, `filter+count`, `filter+count`, `groupBy+collect`
- Version 2 packs 4 of those into one `.agg()` — 2 actions total, same results
- `when().otherwise()` = `CASE WHEN ... THEN ... ELSE` in SQL — same logic, native Spark
- The difference scales on real clusters: S3 reads are slow, doing them 5× is expensive

In [5]:
def sleep_report_v1(df):
    t = time.time()
    toddlers = df.filter(col('age_months') <= 36)

    total      = toddlers.count()                                               # scan 1
    avg_sleep  = toddlers.agg(avg('sleep_hours')).collect()[0][0]               # scan 2
    no_bedtime = toddlers.filter(col('has_regular_bedtime') == 'No').count()    # scan 3
    inadequate = toddlers.filter(col('adequate_sleep') == 'No').count()         # scan 4
    by_region  = toddlers.groupBy('census_region').agg(avg('sleep_hours')).collect()  # scan 5

    return time.time() - t, total, avg_sleep, no_bedtime, inadequate, len(by_region)

t_v1, v1_total, v1_avg, v1_bed, v1_inad, v1_reg = sleep_report_v1(df)
show('VERSION 1 — 5 separate actions (5 full scans)', [
    'Children in survey:    <b style="color:#1e293b">{:,}</b>'.format(v1_total),
    'Avg sleep hours:       <b style="color:#1e293b">{:.1f} h</b>'.format(v1_avg),
    'No regular bedtime:    <b style="color:#1e293b">{:,}</b>'.format(v1_bed),
    'Not getting enough:    <b style="color:#1e293b">{:,}</b>'.format(v1_inad),
    'Census regions:        <b style="color:#1e293b">{}</b>'.format(v1_reg),
    '',
    '⏱  <b style="color:#dc2626">{:.2f}s</b>'.format(t_v1),
], accent='#dc2626')


"Children in survey: 3,083,063"
Avg sleep hours: 10.7 h
"No regular bedtime: 461,714"
"Not getting enough: 1,014,918"
Census regions: 3
""
⏱ 1.92s


In [6]:
def sleep_report_v2(df):
    t = time.time()
    toddlers = df.filter(col('age_months') <= 36)

    # when().otherwise() = CASE WHEN ... THEN ... ELSE in SQL
    # All four metrics in a single aggregation -- 1 action
    m = toddlers.agg(
        count('*').alias('total'),
        avg('sleep_hours').alias('avg_sleep'),
        spark_sum(when(col('has_regular_bedtime') == 'No', 1).otherwise(0)).alias('no_bedtime'),
        spark_sum(when(col('adequate_sleep') == 'No', 1).otherwise(0)).alias('inadequate')
    ).collect()[0]  # collect()[0] = first (only) result row

    by_region = toddlers.groupBy('census_region').agg(avg('sleep_hours')).collect()

    return time.time() - t, m['total'], m['avg_sleep'], m['no_bedtime'], m['inadequate'], len(by_region)

t_v2, v2_total, v2_avg, v2_bed, v2_inad, v2_reg = sleep_report_v2(df)
show('VERSION 2 — 2 actions (4 metrics in 1 aggregation)', [
    'Children in survey:    <b style="color:#1e293b">{:,}</b>'.format(v2_total),
    'Avg sleep hours:       <b style="color:#1e293b">{:.1f} h</b>'.format(v2_avg),
    'No regular bedtime:    <b style="color:#1e293b">{:,}</b>'.format(v2_bed),
    'Not getting enough:    <b style="color:#1e293b">{:,}</b>'.format(v2_inad),
    'Census regions:        <b style="color:#1e293b">{}</b>'.format(v2_reg),
    '',
    '⏱  <b style="color:#16a34a">{:.2f}s</b>'.format(t_v2),
], accent='#16a34a')


"Children in survey: 3,083,063"
Avg sleep hours: 10.7 h
"No regular bedtime: 461,714"
"Not getting enough: 1,014,918"
Census regions: 3
""
⏱ 0.81s


In [7]:
speedup = t_v1 / t_v2
show('COMPARISON', [
    'Version 1 — 5 scans:   <b style="color:#dc2626">{:.2f}s</b>'.format(t_v1),
    'Version 2 — 2 scans:   <b style="color:#16a34a">{:.2f}s</b>'.format(t_v2),
    '',
    '<b style="color:#1e293b">{:.1f}× faster</b> — same numbers, one trip through the data.'.format(speedup),
])


Version 1 — 5 scans: 1.92s
Version 2 — 2 scans: 0.81s
""
"2.4× faster — same numbers, one trip through the data."


---
# Part 3 — The Wellness Score

You want to add three computed metrics: sleep deficit, outdoor-to-screen ratio, a wellness index.  
You add the columns. Then you change your mind — you don't need them for this query.

**Concept: Catalyst column pruning**
- Catalyst reads the entire plan *before* execution
- If a column is created and then dropped without being used, Catalyst skips creating it entirely
- This is "column pruning" — Catalyst prunes columns that don't contribute to the final result

**What to notice (from the table below):**

| | What Spark must do | Speed |
|---|---|---|
| Baseline: `count()` | Nothing extra | Fastest |
| Add 3 cols + aggregate | Must compute all 3 | Slowest |
| Add 3 cols + `drop()` + `count()` | **Catalyst skips creation** | Same as baseline |

- The optional `explain()` cell proves it: the dropped columns simply vanish from the plan
- You didn't configure this — Catalyst did it automatically because it could see the full picture

In [8]:
# Baseline
t = time.time(); 
df.count(); 
t_base = time.time() - t

# Heavy: withColumn() adds a computed column -- like SELECT *, col_a * col_b AS new_col
# These three columns are aggregated, so Spark MUST compute them
t = time.time()
(
    df.withColumn('sleep_deficit',  lit(11.0) - col('sleep_hours'))
        .withColumn('outdoor_ratio',  col('outdoor_play_hours')/(col('screen_time_hours') + 0.5))
        .withColumn('wellness_index', col('sleep_hours') * col('outdoor_play_hours') / 11.0)
        .agg(
              spark_sum('sleep_deficit'),
              spark_sum('outdoor_ratio'),
              spark_sum('wellness_index')
          ).collect()
)
t_heavy = time.time() - t

# Catalyst: same 3 columns, dropped before count()
# drop() removes columns -- Catalyst sees this and skips computing them entirely
t = time.time()
(
    df.withColumn('sleep_deficit',  lit(11.0) - col('sleep_hours'))
        .withColumn('outdoor_ratio',  col('outdoor_play_hours')/(col('screen_time_hours') + 0.5))
        .withColumn('wellness_index', col('sleep_hours') * col('outdoor_play_hours') / 11.0)
        .drop('sleep_deficit', 'outdoor_ratio', 'wellness_index')
        .count()
)
t_catalyst = time.time() - t

show('CATALYST OPTIMISER — column pruning', [
    'Baseline count:                  <b style="color:#1e293b">{:.2f}s</b>'.format(t_base),
    'Add 3 cols + aggregate:          <b style="color:#dc2626">{:.2f}s</b>'.format(t_heavy),
    'Add 3 cols + drop + count:       <b style="color:#16a34a">{:.2f}s</b>  ← same as baseline'.format(t_catalyst),
    '',
    '<b style="color:#1e293b">{:.1f}×</b> faster to create and drop than to create and use.'.format(t_heavy / t_catalyst),
    'Catalyst saw the full plan before running anything — the columns were never computed.',
])


Baseline count: 0.20s
Add 3 cols + aggregate: 0.37s
Add 3 cols + drop + count: 0.17s ← same as baseline
""
2.2× faster to create and drop than to create and use.
Catalyst saw the full plan before running anything — the columns were never computed.


In [9]:
# The plan-level proof
q_heavy = (
    df.withColumn('sleep_deficit',  lit(11.0) - col('sleep_hours'))
        .withColumn('outdoor_ratio',  col('outdoor_play_hours') / (col('screen_time_hours') + 0.5))
        .withColumn('wellness_index', col('sleep_hours') * col('outdoor_play_hours') / 11.0)
        .agg(
            spark_sum('sleep_deficit'), 
            spark_sum('outdoor_ratio'), 
            spark_sum('wellness_index')
        )
)

q_catalyst = (
    df.withColumn('sleep_deficit',  lit(11.0) - col('sleep_hours'))
        .withColumn('outdoor_ratio',  col('outdoor_play_hours') / (col('screen_time_hours') + 0.5))
        .withColumn('wellness_index', col('sleep_hours') * col('outdoor_play_hours') / 11.0)
        .drop('sleep_deficit', 'outdoor_ratio', 'wellness_index')
)

show('PLAN: add 3 cols + aggregate  →  sleep_deficit / outdoor_ratio / wellness_index appear', [], accent='#dc2626')
q_heavy.explain()

show('PLAN: add 3 cols + drop  →  all three are gone (pruned before execution)', [], accent='#16a34a')
q_catalyst.explain()


== Physical Plan ==
*(2) HashAggregate(keys=[], functions=[sum(sleep_deficit#487), sum(outdoor_ratio#507), sum(wellness_index#528)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=510]
   +- *(1) HashAggregate(keys=[], functions=[partial_sum(sleep_deficit#487), partial_sum(outdoor_ratio#507), partial_sum(wellness_index#528)])
      +- *(1) Project [(11.0 - sleep_hours#7) AS sleep_deficit#487, (outdoor_play_hours#11 / (screen_time_hours#10 + 0.5)) AS outdoor_ratio#507, ((sleep_hours#7 * outdoor_play_hours#11) / 11.0) AS wellness_index#528]
         +- *(1) ColumnarToRow
            +- FileScan parquet [sleep_hours#7,screen_time_hours#10,outdoor_play_hours#11] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/app/data/nsch_extended], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sleep_hours:double,screen_time_hours:double,outdoor_play_hours:double>




== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [child_id#0L,survey_year#1,age_months#2,sex#3,state#4,census_region#5,health_status#6,sleep_hours#7,adequate_sleep#8,has_regular_bedtime#9,screen_time_hours#10,outdoor_play_hours#11,reads_together#12,flourishing_affection#13,flourishing_resilience#14,flourishing_curiosity#15,daycare_attendance#16,developmental_concern#17] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/app/data/nsch_extended], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<child_id:bigint,survey_year:int,age_months:int,sex:string,state:string,census_region:strin...




---
# Part 4 — What Is Spark Planning?

`explain()` shows you the execution plan before anything runs. Call it on any DataFrame.

**This is Demo 1's use of explain(): validation — did Catalyst do what I expected?**

**How to read the plan (bottom-up):**
```
Sort / HashAggregate       ← your result
+- Exchange                ← SHUFFLE (data moving across the network = stage boundary)
   +- HashAggregate        ← partial aggregation (one per partition, before shuffle)
      +- Filter            ← your .filter() calls, combined by Catalyst into one
         +- FileScan       ← where data is read; PushedFilters = conditions at file level
```

**What we're confirming here:**
- Four `.filter()` calls in the Parsed plan → one Filter in the Optimized plan: Catalyst combined them
- `PushedFilters` present → conditions are being applied at the file level (predicate pushdown working)
- Write code for readability — Catalyst will optimise it either way

**Tip:** `explain(extended=True)` shows all 4 plan stages. Compare Parsed vs Optimized to see what Catalyst changed.

**Note:** We'll come back to `explain()` in Demo 3 with a different job — spotting problems rather than validating optimisations. Same tool, different question.

In [10]:
# Four separate .filter() calls
q = df \
    .filter(col('census_region') == 'Northeast') \
    .filter(col('age_months') <= 36) \
    .filter(col('adequate_sleep') == 'No') \
    .filter(col('has_regular_bedtime') == 'No')

show('FILTER COMBINING — 4 .filter() calls → 1 in the optimized plan', [
    'Parsed Logical Plan:    4 separate Filter nodes (as written)',
    'Optimized Logical Plan: 1 combined Filter expression (Catalyst merged them)',
    '',
    'Use extended=True to see all 4 plan stages side by side.',
])
q.explain(extended=True)

Parsed Logical Plan: 4 separate Filter nodes (as written)
Optimized Logical Plan: 1 combined Filter expression (Catalyst merged them)
""
Use extended=True to see all 4 plan stages side by side.


== Parsed Logical Plan ==
'Filter ('has_regular_bedtime = No)
+- Filter (adequate_sleep#8 = No)
   +- Filter (age_months#2 <= 36)
      +- Filter (census_region#5 = Northeast)
         +- Relation [child_id#0L,survey_year#1,age_months#2,sex#3,state#4,census_region#5,health_status#6,sleep_hours#7,adequate_sleep#8,has_regular_bedtime#9,screen_time_hours#10,outdoor_play_hours#11,reads_together#12,flourishing_affection#13,flourishing_resilience#14,flourishing_curiosity#15,daycare_attendance#16,developmental_concern#17] parquet

== Analyzed Logical Plan ==
child_id: bigint, survey_year: int, age_months: int, sex: string, state: string, census_region: string, health_status: string, sleep_hours: double, adequate_sleep: string, has_regular_bedtime: string, screen_time_hours: double, outdoor_play_hours: double, reads_together: string, flourishing_affection: string, flourishing_resilience: string, flourishing_curiosity: string, daycare_attendance: string, developmental_concern: string
Filter (ha

In [11]:
# Aggregation plan with Exchange nodes visible
sleep_by_region = df \
    .filter(col('age_months') <= 36) \
    .groupBy('census_region') \
    .agg(
        avg('sleep_hours').alias('avg_sleep'),
        spark_sum(when(col('adequate_sleep') == 'Yes', 1).otherwise(0)).alias('adequate'),
        count('*').alias('total')
    ) \
    .orderBy(col('avg_sleep').desc())

show('AGGREGATION PLAN — look for Exchange nodes (= shuffles)', [
    'FileScan → Filter → HashAggregate → Exchange → HashAggregate → Sort',
    '',
    'Each Exchange = data moving across the network = stage boundary.',
    'More Exchange nodes = more shuffles = more potential for slowness.',
])
sleep_by_region.explain(mode='formatted')

FileScan → Filter → HashAggregate → Exchange → HashAggregate → Sort
""
Each Exchange = data moving across the network = stage boundary.
More Exchange nodes = more shuffles = more potential for slowness.


== Physical Plan ==
* Sort (9)
+- Exchange (8)
   +- * HashAggregate (7)
      +- Exchange (6)
         +- * HashAggregate (5)
            +- * Project (4)
               +- * Filter (3)
                  +- * ColumnarToRow (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [age_months#2, census_region#5, sleep_hours#7, adequate_sleep#8]
Batched: true
Location: InMemoryFileIndex [file:/app/data/nsch_extended]
PushedFilters: [IsNotNull(age_months), LessThanOrEqual(age_months,36)]
ReadSchema: struct<age_months:int,census_region:string,sleep_hours:double,adequate_sleep:string>

(2) ColumnarToRow [codegen id : 1]
Input [4]: [age_months#2, census_region#5, sleep_hours#7, adequate_sleep#8]

(3) Filter [codegen id : 1]
Input [4]: [age_months#2, census_region#5, sleep_hours#7, adequate_sleep#8]
Condition : (isnotnull(age_months#2) AND (age_months#2 <= 36))

(4) Project [codegen id : 1]
Output [3]: [census_region#5, sleep_hours#7, adequate_sleep#8]
Input [4]: [age_mon

---
# 🟢 DEMO 2 — Partitioning

**What we’re demonstrating:**
- On-disk partitioning with `partitionBy('state')`
- Partition pruning — Spark skips entire folders it doesn’t need
- Why the partition key matters: filtering on `state` vs `census_region`
- `repartition` vs `coalesce` — reshaping in-memory partitions
- Data skew and salting — when one partition drowns the rest

**Key thing to watch:**
- Filtering on a non-partition column: `PartitionFilters: []` in the plan — full scan
- Filtering on the partition key: `PartitionFilters: [state = NY]` — 49 folders skipped
- `repartition(n)` does a full shuffle — use before heavy computation
- `coalesce(n)` merges adjacent partitions — use before writing output
- Skew: one executor gets 4.5M rows; salting spreads it across 8


---
# Part 5 — Why Are We Scanning All 50 States?

You keep asking about Northeast data. Every query reads California, Texas, Oregon...  
You reorganize: write the data partitioned by `state`.

**Concept: On-disk partitioning + partition pruning**
- `df.write.partitionBy('state')` creates one folder per state value
- When Spark reads the data back, it knows exactly which folder has which state
- Filter on the partition column → Spark opens only the relevant folder(s)
- Filter on a non-partition column → Spark must open all folders and check

**What to notice:**
- After writing: 50 folders, `state=NY/`, `state=CA/`, etc.
- Non-partition filter (`census_region == 'Northeast'`): `PartitionFilters: []` — full scan
- Partition filter (`state == 'NY'`): `PartitionFilters: [state = NY]` — 49 folders skipped

```
nsch_by_state/
  state=AK/  state=AL/  state=AR/  ...  state=NY/  ...  state=WY/
  (50 folders total — query one, skip 49)
```

**Practical rule:** Choose your partition column based on your most common query filters.  
Bad partition key = partition pruning never triggers = you're reading everything every time.

In [25]:
if not os.path.exists(PART):
    t = time.time()
    df.write.mode('overwrite').partitionBy('state').parquet(PART)
    show('PARTITION WRITE — done', [
        'Written to:  {}'.format(PART),
        'Time:        <b style="color:#16a34a">{:.1f}s</b>'.format(time.time() - t),
    ], accent='#16a34a')
else:
    show('PARTITION WRITE — already exists', [
        'Data already at  {}'.format(PART),
    ])

folders = sorted([f for f in os.listdir(PART) if f.startswith('state=')])
show('ON-DISK STRUCTURE — {} state folders'.format(len(folders)),
    ['  ' + f + '/' for f in folders[:8]] + ['  ... ({} states total)'.format(len(folders))])

Data already at /app/data/nsch_by_state


state=AK/
state=AL/
state=AR/
state=AZ/
state=CA/
state=CO/
state=CT/
state=DE/
... (50 states total)


In [26]:
df_states = spark.read.parquet(PART)

t = time.time()
n = df_states.filter(col('census_region') == 'Northeast').count()
t_full = time.time() - t

show('NON-PARTITION FILTER — census_region (full scan)', [
    'Records found:  <b style="color:#1e293b">{:,}</b>'.format(n),
    'Time:           <b style="color:#dc2626">{:.2f}s</b>'.format(t_full),
    '',
    'PartitionFilters: []  ← Spark opened all 50 folders.',
    'census_region is not the partition key — no pruning possible.',
], accent='#dc2626')
df_states.filter(col('census_region') == 'Northeast').explain()

"Records found: 899,760"
Time: 6.57s
""
PartitionFilters: [] ← Spark opened all 50 folders.
census_region is not the partition key — no pruning possible.


== Physical Plan ==
*(1) Filter (isnotnull(census_region#1424) AND (census_region#1424 = Northeast))
+- *(1) ColumnarToRow
   +- FileScan parquet [child_id#1420L,survey_year#1421,age_months#1422,sex#1423,census_region#1424,health_status#1425,sleep_hours#1426,adequate_sleep#1427,has_regular_bedtime#1428,screen_time_hours#1429,outdoor_play_hours#1430,reads_together#1431,flourishing_affection#1432,flourishing_resilience#1433,flourishing_curiosity#1434,daycare_attendance#1435,developmental_concern#1436,state#1437] Batched: true, DataFilters: [isnotnull(census_region#1424), (census_region#1424 = Northeast)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/app/data/nsch_by_state], PartitionFilters: [], PushedFilters: [IsNotNull(census_region), EqualTo(census_region,Northeast)], ReadSchema: struct<child_id:bigint,survey_year:int,age_months:int,sex:string,census_region:string,health_stat...




In [27]:
t = time.time()
n = df_states.filter(col('state') == 'NY').count()
t_pruned = time.time() - t

show('PARTITION FILTER — state (partition key)', [
    'Records found:  <b style="color:#1e293b">{:,}</b>'.format(n),
    'Time:           <b style="color:#16a34a">{:.2f}s</b>'.format(t_pruned),
    '',
    'PartitionFilters: [state = NY]  ← 49 folders never opened.',
    'Spark went straight to state=NY/ and read only that folder.',
], accent='#16a34a')
df_states.filter(col('state') == 'NY').explain()

"Records found: 99,637"
Time: 0.13s
""
PartitionFilters: [state = NY] ← 49 folders never opened.
Spark went straight to state=NY/ and read only that folder.


== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [child_id#1420L,survey_year#1421,age_months#1422,sex#1423,census_region#1424,health_status#1425,sleep_hours#1426,adequate_sleep#1427,has_regular_bedtime#1428,screen_time_hours#1429,outdoor_play_hours#1430,reads_together#1431,flourishing_affection#1432,flourishing_resilience#1433,flourishing_curiosity#1434,daycare_attendance#1435,developmental_concern#1436,state#1437] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/app/data/nsch_by_state], PartitionFilters: [isnotnull(state#1437), (state#1437 = NY)], PushedFilters: [], ReadSchema: struct<child_id:bigint,survey_year:int,age_months:int,sex:string,census_region:string,health_stat...




---
# Part 6 — Reshaping Partitions

Now that the data is organized on disk, two questions come up about in-memory partitioning:
how many partitions should we have, and how do we change that number?

**Concept: Partition sizing**

| Tool | What it does | Cost | Use when |
|------|--------------|------|----------|
| `repartition(n)` | Full shuffle, perfectly even | Expensive | Before heavy computation |
| `coalesce(n)` | Merge adjacent, no shuffle | Cheap | Before writing output |

* `repartition` can increase or decrease partition count, always shuffles.  
* `coalesce` can only decrease, avoids shuffle — but result may be uneven.

**The shuffle.partitions default:**  
* Default is 200, set in 2013 for large Hadoop clusters.  
* Check your actual core count with `spark.sparkContext.defaultParallelism`.  
* Rule of thumb: 2–3 partitions per core. On this machine: 6 cores → 12 partitions.


In [28]:
cores = spark.sparkContext.defaultParallelism
show('CLUSTER CORES', [
    'spark.sparkContext.defaultParallelism:  <b style="color:#1e293b">{}</b>'.format(cores),
    '',
    'Rule of thumb: 2–3 partitions per core.',
    'This machine: {} cores → {} to {} shuffle partitions'.format(cores, cores*2, cores*3),
])


spark.sparkContext.defaultParallelism: 6
""
Rule of thumb: 2–3 partitions per core.
This machine: 6 cores → 12 to 18 shuffle partitions


In [29]:
show('CURRENT PARTITIONS', [
    'df has <b style="color:#1e293b">{}</b> partition(s) — each maps to one task on one core'.format(df.rdd.getNumPartitions()),
])

# repartition: full shuffle -- redistributes data evenly across N partitions
t = time.time()
df_16 = df.repartition(16)
df_16.count()
t_r = time.time() - t

# coalesce: no shuffle -- merges adjacent partitions, use before writing output
t = time.time()
df_4 = df.coalesce(4)
df_4.count()
t_c = time.time() - t

show('REPARTITION vs COALESCE', [
    'repartition(16):  <b style="color:#1e293b">{} parts</b> | <b style="color:#dc2626">{:.2f}s</b>  ← full shuffle, evenly distributed'.format(df_16.rdd.getNumPartitions(), t_r),
    'coalesce(4):      <b style="color:#1e293b">{} parts</b> | <b style="color:#16a34a">{:.2f}s</b>  ← no shuffle, faster, possibly uneven'.format(df_4.rdd.getNumPartitions(), t_c),
    '',
    'Rule:  repartition before heavy computation — you want balance.',
    '       coalesce before writing output — fewer files, no shuffle cost.',
])


df has 6 partition(s) — each maps to one task on one core


"repartition(16): 16 parts | 0.40s ← full shuffle, evenly distributed"
"coalesce(4): 4 parts | 0.18s ← no shuffle, faster, possibly uneven"
""
Rule: repartition before heavy computation — you want balance.
"coalesce before writing output — fewer files, no shuffle cost."


In [31]:
# Shuffle partition tuning
spark.conf.set('spark.sql.shuffle.partitions', '200')
t = time.time()
df.groupBy('state').agg(avg('sleep_hours')).collect()
t_200 = time.time() - t

spark.conf.set('spark.sql.shuffle.partitions', '12')  # 2 × 6 cores
t = time.time()
df.groupBy('state').agg(avg('sleep_hours')).collect()
t_12 = time.time() - t

show('SHUFFLE PARTITION TUNING', [
    '200 partitions (default):   <b style="color:#dc2626">{:.2f}s</b>'.format(t_200),
    ' 12 partitions (2×cores):  <b style="color:#16a34a">{:.2f}s</b>'.format(t_12),
    '',
    '<b style="color:#1e293b">{:.1f}×</b> faster with right-sized partitions on this machine.'.format(t_200 / t_12),
    '200 was the default set in 2013. 6 cores × 2 = 12 partitions.',
])


200 partitions (default): 0.45s
12 partitions (2×cores): 0.22s
""
2.0× faster with right-sized partitions on this machine.
200 was the default set in 2013. 6 cores × 2 = 12 partitions.


---
# 🔴 DEMO 3 — Caching & UDFs

**What we’re demonstrating:**
- Caching: load data once, serve repeated queries from memory
- UDFs vs built-in functions: what Catalyst can and can’t see

**Key thing to watch:**
- `.cache()` makes the 2nd and 3rd action hit memory instead of disk
- `BatchEvalPython` in UDF plan vs clean CASE expression in built-in plan
- `explain()` as a diagnostic tool — spotting the exact problem in the plan


---
# Part 7 — Three Questions, One Dataset

You want to analyse Northeast toddlers from three different angles.  
Without caching, Spark re-reads the data for each question.  
With `.cache()`, the first action loads it into memory — the rest are served from RAM.

**Concept: Caching**
- `.cache()` tells Spark: after computing this once, hold the result in memory
- Subsequent actions on the same cached DataFrame skip the disk read entirely
- `.unpersist()` releases the memory when you're done — important on shared clusters
- Caching keeps lineage: if a cached partition is evicted, Spark can recompute from source

**What to notice:**
- Without cache: 3 actions = 3 filter passes + 3 disk reads
- With cache: the first `.count()` loads data into memory; the next two are served from RAM
- The speedup depends on your storage: remote S3 reads are slow, memory is fast
- Always `.unpersist()` after the last action — don't hold memory you no longer need

In [33]:
ne_toddlers = df.filter(col('census_region') == 'Northeast').filter(col('age_months') <= 36)

t = time.time()
n           = ne_toddlers.count()                                              # disk read 1
avg_s       = ne_toddlers.agg(avg('sleep_hours')).collect()[0][0]             # disk read 2
pct_bedtime = ne_toddlers.filter(col('has_regular_bedtime') == 'Yes').count() / n  # disk read 3
t_nocache = time.time() - t

show('WITHOUT cache — 3 separate disk reads', [
    'Toddlers (NE, ≤36 mo):  <b style="color:#1e293b">{:,}</b>'.format(n),
    'Avg sleep hours:        <b style="color:#1e293b">{:.1f} h</b>'.format(avg_s),
    'Regular bedtime:        <b style="color:#1e293b">{:.0%}</b>'.format(pct_bedtime),
    '',
    'Time: <b style="color:#dc2626">{:.2f}s</b>  (3 actions = 3 full scans)'.format(t_nocache),
], accent='#dc2626')

"Toddlers (NE, ≤36 mo): 899,760"
Avg sleep hours: 10.7 h
Regular bedtime: 85%
""
Time: 0.73s (3 actions = 3 full scans)


In [34]:
ne_toddlers_c = df.filter(col('census_region') == 'Northeast').filter(col('age_months') <= 36)
ne_toddlers_c.cache()  # loads into memory on first action

t = time.time()
n           = ne_toddlers_c.count()                                               # loads cache
avg_s       = ne_toddlers_c.agg(avg('sleep_hours')).collect()[0][0]              # from memory
pct_bedtime = ne_toddlers_c.filter(col('has_regular_bedtime') == 'Yes').count() / n  # from memory
t_cache = time.time() - t

show('WITH cache — loaded once, served from memory', [
    'Toddlers (NE, ≤36 mo):  <b style="color:#1e293b">{:,}</b>'.format(n),
    'Avg sleep hours:        <b style="color:#1e293b">{:.1f} h</b>'.format(avg_s),
    'Regular bedtime:        <b style="color:#1e293b">{:.0%}</b>'.format(pct_bedtime),
    '',
    'Time:    <b style="color:#16a34a">{:.2f}s</b>'.format(t_cache),
    'Speedup: <b style="color:#16a34a">{:.1f}×</b> — 2nd and 3rd actions read from RAM'.format(t_nocache / t_cache),
], accent='#16a34a')

ne_toddlers_c.unpersist()  # free the memory when done
show('CACHE RELEASED', ['ne_toddlers_c.unpersist() — memory freed.'])

"Toddlers (NE, ≤36 mo): 899,760"
Avg sleep hours: 10.7 h
Regular bedtime: 85%
""
Time: 1.92s
Speedup: 0.4× — 2nd and 3rd actions read from RAM


ne_toddlers_c.unpersist() — memory freed.


---
# Part 8 — The Sleep Classifier

You want to label each child: well_rested, borderline, or sleep_deprived.  
You write a Python function. It works — but Catalyst can no longer see inside it.

**This is Demo 3's use of explain(): diagnosis — what is slowing this down?**

**Concept: UDFs vs built-in functions**

| | UDF (`@udf`) | Built-in (`when().otherwise()`) |
|--|--|--|
| Catalyst visibility | Black box | Fully visible |
| Optimization | None | Pushdown, pruning, combining |
| Execution | Row-by-row in Python process | Vectorized in JVM |
| Plan shows | `BatchEvalPython` | CASE expression in Project node |

**What to look for in `explain()` to diagnose pitfalls:**
- `BatchEvalPython` → UDF is blocking optimization — replace with built-in if possible
- `PushedFilters: []` on a filtered column → predicate not reaching the data source — likely a UDF or complex expression in the filter chain
- More `Exchange` nodes than expected → unnecessary shuffles — check partition key and groupBy logic
- Same FileScan appearing twice → missing a cache on reused data

**Practical rule:** When a job is slow and you don't know why, run `explain()` first. The plan tells you exactly where to look — before you start guessing.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Python UDF -- opaque to Catalyst
@udf(returnType=StringType())
def sleep_category_udf(hours):
    if hours >= 11:
        return 'well_rested'
    elif hours >= 9:
        return 'borderline'
    else:
        return 'sleep_deprived'

# Built-in when() -- Catalyst sees every condition
sleep_category_builtin = when(col('sleep_hours') >= 11, 'well_rested') \
    .when(col('sleep_hours') >= 9, 'borderline') \
    .otherwise('sleep_deprived')

show('UDF PLAN — Catalyst sees a black box', [
    'Look for:  BatchEvalPython  ← opaque to Catalyst',
    'Cannot push down, combine, or optimize. Serializes data to Python row-by-row.',
], accent='#dc2626')
df.withColumn('sleep_cat', sleep_category_udf(col('sleep_hours'))).explain()

show('BUILT-IN PLAN — Catalyst sees the full CASE expression', [
    'Look for:  CASE WHEN in the Project node  ← fully visible to Catalyst',
    'Same output. Catalyst can optimize, push down, and combine freely.',
], accent='#16a34a')
df.withColumn('sleep_cat', sleep_category_builtin).explain()

---
# Part 9 — California Is Breaking Your groupBy

In the real NSCH, California has roughly 10% of all child records — more than any other state.  
Simulating an extreme version: 90% of records land in one state.

When Spark shuffles this data for a `groupBy('state')`,  
one partition gets 4.5 million rows. 49 partitions share 500K.  
One executor does all the work. Your job runs as long as the slowest task.

Fix: **salting** — add a random prefix to split the hot key across multiple partitions.

In [23]:
# 90% of records become California
skewed = df.withColumn('state_skewed',
    when(rand() > 0.9, col('state')).otherwise(lit('CA'))
)

show('SKEWED DATA — 90% of records → California', [
    'Top 5 states by record count (Spark .show() output below):',
])
skewed.groupBy('state_skewed').count().orderBy(col('count').desc()).show(5)

t = time.time()
skewed.groupBy('state_skewed').agg(avg('sleep_hours')).collect()
t_skewed = time.time() - t

show('SKEWED groupBy — result', [
    'Time: <b style="color:#dc2626">{:.2f}s</b>'.format(t_skewed),
    '',
    'In Spark UI: CA task = 4.5M rows. All others = ~10K each.',
    'One executor drowning. The rest idle. Job runs as long as the slowest task.',
], accent='#dc2626')

Top 5 states by record count (Spark .show() output below):


+------------+-------+
|state_skewed|  count|
+------------+-------+
|          CA|4509495|
|          MS|  10276|
|          OK|  10190|
|          AR|  10186|
|          UT|  10167|
+------------+-------+
only showing top 5 rows



Time: 0.47s
""
In Spark UI: CA task = 4.5M rows. All others = ~10K each.
One executor drowning. The rest idle. Job runs as long as the slowest task.


In [24]:
from pyspark.sql.functions import concat, regexp_replace

spark.conf.set('spark.sql.shuffle.partitions', '8')
N_SALT = 8

# Add random prefix 0-7 to split CA into 8 separate keys
salted = skewed \
    .withColumn('salt', (rand() * N_SALT).cast('int')) \
    .withColumn('state_salted', concat(col('salt').cast('string'), lit('_'), col('state_skewed')))

# Phase 1: group by salted key -- now evenly distributed
pre = salted.groupBy('state_salted').agg(
    spark_sum('sleep_hours').alias('total_sleep'),
    count('*').alias('n')
)

# Phase 2: strip salt, combine partial results
final = pre \
    .withColumn('state_skewed', regexp_replace(col('state_salted'), '^[0-9]+_', '')) \
    .groupBy('state_skewed').agg(
        (spark_sum('total_sleep') / spark_sum('n')).alias('avg_sleep')
    )

t = time.time()
final.collect()
t_salted = time.time() - t

show('SALTING — skew eliminated', [
    'Time: <b style="color:#16a34a">{:.2f}s</b>'.format(t_salted),
    '',
    'CA is now 0_CA, 1_CA ... 7_CA — spread evenly across 8 partitions.',
    'Two groupBys instead of one, but all executors contribute equally.',
    '',
    'Spark 3.0+ AQE handles mild skew automatically.',
    'For extreme skew — salting is still the most reliable fix.',
], accent='#16a34a')

Time: 0.44s
""
"CA is now 0_CA, 1_CA ... 7_CA — spread evenly across 8 partitions."
"Two groupBys instead of one, but all executors contribute equally."
""
Spark 3.0+ AQE handles mild skew automatically.
For extreme skew — salting is still the most reliable fix.


In [ ]:
spark.stop()
print('Session closed.')